In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH08-Embeddings")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


- 비용 효율은 1달러당 몇 페이지의 임베딩을 할 수 있는지
- MTEB는 임베딩 모델의 성능을 평가하기 위한 대규모 벤치마크

지원되는 모델 목록

| MODEL                  | PAGES PER DOLLAR | PERFORMANCE ON MTEB EVAL | MAX INPUT |
|------------------------|------------------|---------------------------|-----------|
| text-embedding-3-small | 62,500           | 62.3%                     | 8191      |
| text-embedding-3-large | 9,615            | 64.6%                     | 8191      |
| text-embedding-ada-002 | 12,500           | 61.0%                     | 8191      |

In [3]:
from langchain_openai import OpenAIEmbeddings

# OpenAI의 "text-embedding-3-small" 모델을 사용하여 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."

- `embeddings.embed_query(text)`는 주어진 텍스트를 임베딩 벡터로 변환하는 함수
- 텍스트를 벡터 공간에 매핑하여 의미적으로 유사한 텍스트를 찾거나 텍스트 간의 유사도를 계산하는 데 사용

In [5]:
# 텍스트를 임베딩하여 쿼리 결과를 생성합니다.
query_result = embeddings.embed_query(text)

- 결과를 보면 1536이 출력되는데 차원의 개수로 보면 됨

In [7]:
len(query_result)
# 쿼리 결과의 처음 5개 항목을 선택
query_result[:5]

[-0.007778701838105917,
 0.036736391484737396,
 0.019512122496962547,
 -0.019740907475352287,
 0.017256952822208405]

- `embeddings.embed_documents()` 함수를 사용하여 텍스트 문서를 임베딩
- `[text]`를 인자로 전달하여 단일 문서를 리스트 형태로 임베딩 함수에 전달
- 함수 호출 결과로 반환된 임베딩 벡터를 `doc_result` 변수에 할당

In [9]:
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)  # 텍스트를 임베딩하여 문서 벡터를 생성

- `doc_result[0][:5]`는 `doc_result` 리스트의 첫 번째 요소에서 처음 5개의 문자를 슬라이싱하여 선택

In [12]:
len(doc_result)  # 문서 벡터의 길이를 확인
# 문서 결과의 첫 번째 요소에서 처음 5개 항목을 선택합니다.
doc_result[0][:5]

[-0.007778701838105917,
 0.036736391484737396,
 0.019512122496962547,
 -0.019740907475352287,
 0.017256952822208405]

### 차원 지정

- 임베딩을 생성할 때 차원 조절 가능. 벡터 데이터베이스가 지원하는 임베딩 차원이 우리가 생성한 임베딩 차원과 다를 경우 유용
- `text-embedding-3` 모델 클래스를 사용하면 반환되는 임베딩의 크기를 지정 가능
- 예를 들어, 기본적으로 `text-embedding-3-small`는 1536 차원의 임베딩을 반환

In [13]:
# 문서 결과의 첫 번째 요소의 길이를 반환
len(doc_result[0])

1536

### 차원(dimensions) 조정

- `dimensions=1024`를 전달함으로써 임베딩의 크기를 1024로 줄일 수 있다

In [14]:
# OpenAI의 "text-embedding-3-small" 모델을 사용하여 1024차원의 임베딩을 생성하는 객체를 초기화
embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

In [15]:
# 주어진 텍스트를 임베딩하고 첫 번째 임베딩 벡터의 길이를 반환
len(embeddings_1024.embed_documents([text])[0])

1024

### 유사도 계산

In [16]:
sentence1 = "안녕하세요? 반갑습니다."
sentence2 = "안녕하세요? 반갑습니다!"
sentence3 = "안녕하세요? 만나서 반가워요."
sentence4 = "Hi, nice to meet you."
sentence5 = "I like to eat apples."

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings_1024.embed_documents(sentences)

- 코사인 유사도 계산
- -1과 1사이 값이며, 1에 가까울 수록 두 벡터가 더 유사

In [19]:
def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [20]:
# sentence1 = "안녕하세요? 반갑습니다."
# sentence2 = "안녕하세요? 만나서 반가워요."
# sentence3 = "Hi, nice to meet you."
# sentence4 = "I like to eat apples."

for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )

[유사도 0.9644] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 반갑습니다!
[유사도 0.8376] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.5043] 안녕하세요? 반갑습니다. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1362] 안녕하세요? 반갑습니다. 	 <=====> 	 I like to eat apples.
[유사도 0.8143] 안녕하세요? 반갑습니다! 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.4791] 안녕하세요? 반갑습니다! 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1318] 안녕하세요? 반갑습니다! 	 <=====> 	 I like to eat apples.
[유사도 0.5128] 안녕하세요? 만나서 반가워요. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1408] 안녕하세요? 만나서 반가워요. 	 <=====> 	 I like to eat apples.
[유사도 0.2250] Hi, nice to meet you. 	 <=====> 	 I like to eat apples.
